# Phase 4: bulk embedding on Colab

Thin wrapper: clones the repo, installs the `embed` extra, runs
`trustlayer.embed.run` with `--skip-load`, saves each chunk file to
Drive as it lands. No logic lives here — the module owns everything.
Interrupt and re-run any time: completed chunks are skipped (resume).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ctru0009/trustlayer.git /content/trustlayer
%cd /content/trustlayer/python
!pip install -q uv
!uv sync --extra embed --extra spark

In [ ]:
import shutil
import subprocess
from pathlib import Path

OUT = Path('/content/trustlayer/data/processed/embeddings')
BACKUP = Path('/content/drive/MyDrive/trustlayer/embeddings')
BACKUP.mkdir(parents=True, exist_ok=True)

# Restore previously saved chunks so the run resumes where it stopped.
for saved in BACKUP.glob('chunk-*.jsonl'):
    shutil.copy(saved, OUT / saved.name) if OUT.is_dir() else None

proc = subprocess.Popen(
    ['uv', 'run', '--extra', 'embed', '--extra', 'spark',
     'python', '-m', 'trustlayer.embed.run', '--skip-load',
     '--device', 'cuda'],
    cwd='/content/trustlayer/python', text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
seen = set(BACKUP.glob('chunk-*.jsonl'))
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
    for fresh in OUT.glob('chunk-*.jsonl'):
        if fresh.name not in {p.name for p in seen}:
            shutil.copy(fresh, BACKUP / fresh.name)
            print(f'[drive] saved {fresh.name}')
    seen = set(BACKUP.glob('chunk-*.jsonl'))
print('exit:', proc.wait())